In [8]:
import os
import pandas as pd
import geopandas as gpd
import numpy as np

In [2]:
# Read shapefile path
shapefile_path= '../../SMM_Models/hype/geospatial/shapefiles/modified_shapefiles/Modified_SMMcat.shp'

In [3]:
bias_corr_directory= '/work/comphyd_lab/users/paul.coderre/Data_2_compact/'

In [4]:
# Define the number you want to search for
search_number = "05"

In [5]:
# Read shapefile
shapefile= gpd.read_file(shapefile_path)

# Convert ID cols to int
shapefile['hru_nhm'] = shapefile['hru_nhm'].astype(int)
shapefile['seg_nhm'] = shapefile['seg_nhm'].astype(int)

# Create dictionary for IDs
id_dict = dict(zip(shapefile['hru_nhm'], shapefile['seg_nhm']))

# List all CSV files in the directory
csv_files = [file for file in os.listdir(bias_corr_directory) if file.endswith('.csv')]

# Find the file that contains the defined number in its name
matching_file = None
for file in csv_files:
    if search_number in file:
        matching_file = file
        break

# Concatenate the full filepath
file_path = os.path.join(bias_corr_directory, matching_file)

# Read the bias corrected forcing
df= pd.read_csv(file_path, index_col=0)

# Convert the index to datetime
df.index = pd.to_datetime(df.index)

# Remove 'Basin_' prefix and convert column names to integers
df.columns = df.columns.str.replace('Basin_', '', regex=False).astype(int)

# Rename headers to HYPE basin ID
df = df.rename(columns=id_dict)

In [6]:
df

,58183,58184,58185,58186,58188,58189,58192,58193,58194,58197,...,58666,58667,58668,58669,58670,58671,58672,58673,58674,58675
Dates,,,,,,,,,,,,,,,,,,,,,
1950-01-01,0.000172,0.000179,0.000181,0.000169,0.000151,0.000133,0.000133,0.000113,0.000090,0.000070,...,0.003804,0.004877,0.003634,0.005290,0.005597,0.004178,0.003377,0.003196,0.000398,0.003763
1950-01-02,0.010925,0.010548,0.010428,0.010729,0.010581,0.011085,0.010565,0.011161,0.011163,0.010524,...,0.000799,0.000428,0.000953,0.000598,0.000582,0.000931,0.001245,0.001787,0.010666,0.000512
1950-01-03,0.009609,0.009945,0.009988,0.010206,0.011061,0.011568,0.011950,0.012427,0.010982,0.009791,...,0.010525,0.013327,0.010797,0.010998,0.011119,0.010631,0.010719,0.010699,0.010506,0.013545
1950-01-04,0.852471,0.852609,0.848845,0.883607,0.966087,1.325378,0.916574,0.000000,0.874504,1.331173,...,0.010316,0.010635,0.010831,0.010405,0.010656,0.010884,0.010770,0.011271,0.050974,0.010660
1950-01-05,2.847641,2.453707,2.408532,2.427371,2.456573,2.069685,2.697458,2.547766,2.555205,2.084452,...,0.432679,0.309563,0.547845,0.537710,0.543692,0.706971,0.844426,1.130666,1.369682,0.277231
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2100-11-20,0.025915,0.027176,0.027181,0.028130,0.030655,0.027853,0.031589,0.025565,0.017713,0.012907,...,0.064612,0.171263,0.079977,0.153188,0.181556,0.124228,0.117492,0.154499,0.064104,0.107337
2100-11-21,0.011711,0.012140,0.012260,0.011859,0.011618,0.011556,0.012147,0.012128,0.012450,0.012592,...,0.012114,0.013773,0.012047,0.012619,0.012535,0.012251,0.011821,0.011776,0.012224,0.014634
2100-11-22,0.011868,0.011518,0.011474,0.011661,0.011667,0.011915,0.012286,0.012491,0.012889,0.013024,...,0.012139,0.011844,0.012533,0.012022,0.011904,0.011889,0.011572,0.011553,0.012225,0.012025


In [9]:

# Create a boolean DataFrame indicating where complex numbers are
complex_mask = df.applymap(np.iscomplex)

# Extract coordinates (row index and column name) where complex values exist
complex_coords = [(row, col) for row, col in zip(*np.where(complex_mask.values))]

# Convert to readable coordinates (index and column labels)
complex_entries = [(df.index[i], df.columns[j]) for i, j in complex_coords]

print(complex_entries)

/tmp/ipykernel_1952822/1152855070.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  complex_mask = df.applymap(np.iscomplex)


KeyboardInterrupt: 

In [6]:
# Remove invalid dates
# Function to check if a date is invalid (Feb 29 or Feb 30)
def is_invalid_date(date):
    try:
        month_day = date.strftime('%m-%d')  # Get MM-DD string
        if month_day in ['02-29', '02-30']:
            print("Invalid date removed:", date)
            return True
        return False
    except Exception as e:
        print("Error parsing date:", date, "-", e)
        return True

# Apply the invalid date check
invalid_mask = df.index.to_series().apply(is_invalid_date)

# Report
num_invalid = invalid_mask.sum()
if num_invalid > 0:
    print(f"{num_invalid} invalid dates removed.")
else:
    print("No invalid dates found.")

# Keep only valid dates
df = df[~invalid_mask].copy()

# Add back missing dates
# Define full expected daily date range
min_date = df.index.min()
max_date = df.index.max()

# Define expected dates for that range
expected_dates = pd.date_range(start=min_date, end=max_date, freq='D')

# Identify the missing dates
missing_dates = expected_dates.difference(df.index)

# Add the missign dates and backfill them
if not missing_dates.empty:
    print(f"{len(missing_dates)} missing dates found. Filling...")

    df = df.reindex(expected_dates)

    # Use the updated, preferred syntax
    df = df.ffill().bfill()

# Rename the index
df.index.name = "time"

# Save to a tab-separated file
# df.to_csv("Pobs.txt", sep="\t")


No invalid dates found.
37 missing dates found. Filling...


In [7]:
df

,58183,58184,58185,58186,58188,58189,58192,58193,58194,58197,...,58666,58667,58668,58669,58670,58671,58672,58673,58674,58675
time,,,,,,,,,,,,,,,,,,,,,
1950-01-01,0.000173,0.000179,0.000181,0.000169,0.000151,0.000133,0.000133,0.000113,0.000090,0.000069,...,0.003812,0.004886,0.003643,0.005298,0.005606,0.004184,0.003375,0.003198,0.000400,0.003762
1950-01-02,0.010871,0.011022,0.011042,0.010809,0.010310,0.010670,0.010420,0.010789,0.010875,0.010265,...,0.000801,0.000428,0.000953,0.000598,0.000582,0.000931,0.001244,0.001788,0.010914,0.000512
1950-01-03,0.009591,0.009938,0.009979,0.010010,0.010384,0.010904,0.011517,0.011878,0.010544,0.009283,...,0.010367,0.012980,0.010750,0.010780,0.010822,0.010556,0.010539,0.010603,0.010383,0.013422
1950-01-04,0.819354,0.763092,0.745978,0.826769,0.926632,0.926936,0.861131,0.000000,0.874583,1.169164,...,0.010209,0.010602,0.010606,0.010406,0.010633,0.010273,0.010532,0.010651,0.053308,0.010479
1950-01-05,2.338518,2.370252,2.350940,2.069070,2.034813,2.203992,2.496841,2.284258,2.157303,1.933479,...,0.448810,0.322813,0.549874,0.561332,0.567187,0.737284,0.835981,1.127032,1.286305,0.297436
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2100-11-20,0.086378,0.099536,0.100207,0.108502,0.135455,0.163389,0.186872,0.181043,0.173653,0.180452,...,0.212525,0.311389,0.229716,0.227387,0.249913,0.200239,0.214230,0.180704,0.250226,0.353636
2100-11-21,0.013315,0.013439,0.013435,0.013285,0.012868,0.012801,0.013798,0.013067,0.013305,0.013415,...,0.282945,0.413094,0.247157,0.321615,0.349493,0.247237,0.240070,0.224310,0.186625,0.393144
2100-11-22,0.284692,0.301759,0.302469,0.301360,0.303627,0.341196,0.311043,0.361189,0.353636,0.301013,...,0.012733,0.012774,0.012647,0.012691,0.012457,0.012918,0.012634,0.012573,0.016586,0.012511
